# EVSC 410 — Riverbed evolution: model background and first experiment

**Main question:** How does a change in upstream sediment supply affect the bed elevation of a gravel-bed river?

This notebook introduces the equations used in a simple one-dimensional morphodynamic model. We will first connect water flow to sediment transport, and then connect sediment transport to changes in the riverbed. The calculations use a **constant channel width**, a **single representative grain size**, and **steady, normal-flow hydraulics** at each time step.

**Learning goals:** (1) explain the normal-depth approximation; (2) calculate the role of Shields stress in bedload transport; (3) interpret the Exner equation and its numerical form; and (4) predict whether a riverbed will aggrade or degrade when sediment supply changes.

This notebook describes a simplified model of riverbed evolution. The model combines normal-flow hydraulics, a single representative grain size, a Meyer-Peter–Müller-type bedload relation, and sediment continuity. It does not represent backwater effects, mixed-grain transport, or bed-surface sorting.

## 1. Water flow: the normal-depth approximation

For a wide, rectangular channel with constant width $B$, discharge $Q$ is related to mean velocity $U$ and flow depth $h$ by water continuity:

$$Q = BUh, \qquad q_w = \frac{Q}{B}=Uh.$$

Here $q_w$ is water discharge per unit channel width (m$^2$/s). A more general description of steady, gradually varied flow is given by the depth-gradient equation (Parker, 2007):

$$\frac{dh}{dx}=\frac{S-S_f}{1-Fr^2},$$

where $S=-d\eta/dx$ is the local bed slope, $S_f$ is the friction slope, and $Fr=U/\sqrt{gh}$ is the Froude number. **For normal (uniform) flow**, $dh/dx=0$, so $S_f=S$. This avoids solving the backwater equation and is the central hydraulic simplification in our teaching model.

We use a Manning–Strickler resistance relation (Parker, 2007):

$$C_f=\frac{1}{\alpha_r^2}\left(\frac{h}{k_s}\right)^{-1/3}, \qquad S_f=\frac{C_fU^2}{gh},$$

where $k_s$ is roughness height, $C_f$ is a dimensionless resistance coefficient, and $\alpha_r=8.1$. Substitute $U=q_w/h$ and set $S_f=S$. Rearranging gives **normal depth**:

$$\boxed{h_n=\left(\frac{k_s^{1/3}q_w^2}{\alpha_r^2gS}\right)^{3/10}}.$$

The model evaluates this expression using the local, positive bed slope. It then calculates $U=q_w/h_n$. The model does not calculate backwater effects, hydraulic jumps, or flow over adverse (upstream-rising) bed slopes.

**Think about it:** For fixed $Q$, $B$, and roughness, should flow depth increase or decrease as the channel becomes steeper?

## 2. From flow to bedload transport

Under the wide-channel normal-flow approximation, bed shear stress and shear velocity are

$$\tau_b=\rho_w gh_nS, \qquad u_* = \sqrt{gh_nS}.$$

We express the stress relative to the submerged weight of a representative sediment grain using the **Shields parameter**:

$$\boxed{\theta=\frac{\tau_b}{(\rho_s-\rho_w)gD}=\frac{h_nS}{RD}}, \qquad R=\frac{\rho_s}{\rho_w}-1.$$

Here $D$ is the representative grain diameter, $\rho_s$ and $\rho_w$ are sediment and water densities, and $R=1.65$ in the model. Larger $\theta$ generally indicates greater potential for grain motion.

The teaching model uses a **Meyer-Peter–Müller-type excess-Shields relation** (Meyer-Peter and Müller, 1948) for volumetric bedload transport per unit width:

$$\boxed{q_b=3.97\,\sqrt{RgD}\,D\,\max(\theta-\theta_c,0)^{3/2}}.$$

Here $q_b$ has units of m$^2$/s and $\theta_c$ is the critical Shields stress (default $0.045$). The factor $3.97$ is the coefficient implemented in **this teaching model**; published Meyer-Peter–Müller formulations and calibrations may use different coefficients. If $\theta\leq\theta_c$, this equation predicts zero bedload transport.

This single-size formulation does not track individual grain-size fractions, hiding effects, or changes in surface grain-size distribution.

**Think about it:** Why might increasing discharge increase bedload transport much more rapidly than it increases mean flow velocity?

## 3. Sediment continuity: the Exner equation

Bedload entering and leaving a short river segment determines whether sediment accumulates or is removed. The one-dimensional Exner equation is (Exner, 1925; Parker, 2007)

$$\boxed{(1-\lambda_p)\frac{\partial\eta}{\partial t}=-\frac{\partial q_b}{\partial x}},$$

where $\eta$ is bed elevation, $t$ is time, and $\lambda_p$ is bed porosity. The factor $(1-\lambda_p)$ is the **solid fraction** of the bed material.

- If bedload transport **decreases downstream**, $\partial q_b/\partial x<0$: more sediment enters than leaves, so the bed **aggrades** ($\partial\eta/\partial t>0$).
- If bedload transport **increases downstream**, $\partial q_b/\partial x>0$: more sediment leaves than enters, so the bed **degrades** ($\partial\eta/\partial t<0$).
- If transport is the same everywhere, there is no bed elevation change from sediment-flux divergence.

This is a **mass balance**, not a rule that high transport rates necessarily produce erosion. A large but spatially constant $q_b$ produces no change in bed elevation.

**Think about it:** A reach transports a large amount of sediment, but the incoming and outgoing rates are equal. Does its bed elevation change?

## 4. How the computer solves Exner

Using a conservative finite-volume approach (LeVeque, 2002), the river is divided into cells of length $\Delta x$. The model stores bed elevation $\eta_i$ at the **centre** of each cell and computes bedload flux $q_{b,i+1/2}$ at its **faces**. The sediment balance over cell $i$ is

$$ (1-\lambda_p)\,\Delta x\,\frac{d\eta_i}{dt}=q_{b,i-1/2}-q_{b,i+1/2}. $$

Using a forward step in time, the **discretized Exner equation** is

$$\boxed{\eta_i^{n+1}=\eta_i^n-\frac{\Delta t}{(1-\lambda_p)\Delta x}\left(q_{b,i+1/2}^n-q_{b,i-1/2}^n\right)}.$$

The superscript $n$ denotes the time step. Interior-face slopes are calculated from adjacent cell-centre elevations:

$$S_{i+1/2}=\frac{\eta_i-\eta_{i+1}}{\Delta x}.$$

At each time step, the code uses this sequence:

1. Estimate local slopes at cell faces and compute normal depth and Shields stress.
2. Calculate bedload transport capacity at each face.
3. Prescribe the **upstream sediment feed**; use the calculated transport capacity at other faces.
4. Update each cell's bed elevation using the Exner balance above.

At the downstream end, the model extends the final interior slope to the outlet face. This is a simplified downstream boundary treatment. For a channel of constant width, the change in bed-sediment storage should equal upstream sediment input minus downstream sediment output (accounting for porosity).

The model uses an adaptive time step and checks that slopes remain positive. Its morphodynamic stability estimate is

$$C_m=\frac{D_m\Delta t}{\Delta x^2},\qquad D_m=\frac{1}{1-\lambda_p}\frac{\partial q_b}{\partial S}.$$

The code limits $C_m$ to a conservative value of $0.20$ when selecting the time step. This is a numerical safeguard, **not** proof that the model is accurate or converged. The hydraulic Courant number $C_h=U\Delta t/\Delta x$ is reported for reference; it does not control the quasi-steady hydraulic calculation.

**Think about it:** In the discrete equation, what happens to $\eta_i$ when the upstream-face sediment flux is larger than the downstream-face flux?

## 5. First experiment — changing upstream sediment supply

Before running the model, predict the direction of bed adjustment for sediment feed ratios of **0.7**, **1.0**, and **1.3** relative to the transport capacity of the initial uniform bed. The three runs differ only in upstream sediment supply.

The initial bed is a straight, downstream-sloping profile. The experiment tests the consequences of sediment imbalance rather than reproducing a particular restoration site.

In [ ]:
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
from teaching_model import Parameters, classroom_estimate, simulate

p = Parameters(length_m=300, dx_m=5, width_m=8, initial_slope=.008,
               discharge_m3s=8, grain_m=.025, duration_s=1800,
               max_dt_s=2, save_count=7)


### 5.1 Check numerical settings

Review the initial hydraulic and morphodynamic stability estimates before running the simulations. The model may shorten the time step as the bed evolves.

In [ ]:
d = classroom_estimate(p)
for name in ['cells','depth_m','velocity_ms','hydraulic_courant',
             'morphodynamic_number','max_morphodynamic_dt_s',
             'planned_dt_s','estimated_steps','stored_profiles_kib']:
    print(f'{name:32s} {d[name]:.5g}')
if d['caution']:
    print('CAUTION: large run for a browser. Reduce cells, duration, or saved profiles.')
else:
    print('Initial run-size estimate is suitable for classroom exploration.')


### 5.2 Run three sediment-supply scenarios

Compare the change in bed elevation relative to the initial profile. Positive change means aggradation; negative change means degradation.

In [ ]:
results = {}
for ratio in [0.7, 1.0, 1.3]:
    scenario = replace(p, feed_ratio=ratio)
    info = classroom_estimate(scenario)
    if info['caution']:
        raise ValueError('Reduce run size before proceeding')
    results[ratio] = simulate(scenario)

fig, ax = plt.subplots(figsize=(8,4))
for ratio, r in results.items():
    ax.plot(r['x_m'], r['bed_m'][-1]-r['bed_m'][0], label=f'feed / capacity = {ratio}')
ax.set(xlabel='Distance downstream (m)',ylabel='Bed elevation change (m)',title='Bed adjustment')
ax.legend(); ax.grid(alpha=.25)
plt.show()
for ratio,r in results.items():
    print(f'{ratio:.1f}: steps={r["steps"]}, storage={r["storage_change_m3"]:.6g} m³, '
          f'mass residual={r["mass_balance_error_m3"]:.2e} m³')


## 6. Interpret the results

1. Which scenario aggrades, which degrades, and why? Compare your predictions with the plots.
2. Explain why the largest changes occur near the upstream boundary during these short runs.
3. How small is the sediment mass-balance residual? Does a small residual establish that the model is physically correct?
4. Change **only one** parameter (grain size, width, discharge, or slope). Predict its effect, then rerun the preflight and experiment.
5. Identify two river settings where normal-flow hydraulics would not be appropriate.

**Optional numerical check:** Compare results using $\Delta x=5$ m and $2.5$ m, allowing the model to adjust its time step. Does the predicted bed evolution change?

**Model scope:** The equations and boundary treatments described here correspond to the simplified Python teaching model. The model is intended for exploring physical mechanisms, not predicting changes at a specific restoration site.

## References

- **Exner, F. M. (1925).** Über die Wechselwirkung zwischen Wasser und Geschiebe in Flüssen. *Sitzungsberichte der Akademie der Wissenschaften in Wien, Mathematisch-Naturwissenschaftliche Klasse*, 134, 165–204.
- **LeVeque, R. J. (2002).** *Finite Volume Methods for Hyperbolic Problems*. Cambridge University Press.
- **Meyer-Peter, E., and Müller, R. (1948).** Formulas for bed-load transport. *Proceedings of the 2nd Meeting of the International Association for Hydraulic Structures Research*, Stockholm, 39–64.
- **Parker, G. (2007).** *1D Sediment Transport Morphodynamics with Applications to Rivers and Turbidity Currents*. University of Illinois course notes.

**Note:** The coefficient 3.97 in the bedload equation and the specific boundary and time-stepping choices are those used in this teaching implementation. They should not be interpreted as universal values or as direct quotations from the cited references.
